# Amparo -- M1: reevaluacion de v1 con el prompt correcto

**Solo inferencia. No entrena.**

Las corridas del 2026-10-09 generaron las respuestas de validacion con el system
prompt de `records[0]`, el de sin contexto. Los 103 registros con contexto --
los 35 B2 entre ellos -- se evaluaron **sin la orden de abstenerse** con la que se
entrenaron. Esta corrida vuelve a generar con el adaptador v1, dandole a cada
registro su propio prompt, y **nada mas cambia**: el contexto, la pregunta y el
adaptador son los de entonces (salen de `results/m1_2026-10-09`).

Detalle y regla de lectura: `tools/evaluation/reevaluacion_v1.py` y
`docs/m1_protocolo_aceptacion.md`.

**Coste:** 10 generaciones de control + 103 con contexto, ~35-40 min en A100.


In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate
!pip install -q -U "bitsandbytes>=0.46.1"


In [ ]:
RAMA = "m3.5"

import os

if not os.path.isdir("/content/Amparo"):
    !git clone -q -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd /content/Amparo && git fetch -q origin && git checkout -q {RAMA} && git pull -q
%cd /content/Amparo
!git log -1 --format="Codigo del commit %h (%ad)" --date=short


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Colab Notebooks/Amparo'
# El adaptador de PRODUCCION, el que se entreno el 2026-10-09. Se lee, no se
# escribe: el script comprueba su huella (8ce3cc2bc9306974) y se niega a
# generar si es otro.
ADAPTADOR_V1 = f'{DRIVE_DIR}/amparo-lora-adapter'
SALIDA = f'{DRIVE_DIR}/evaluacion/reevaluacion_v1'


## 1. Plan, sin cargar el modelo

Comprueba el adaptador, la fuente, los prompts y la revision **antes** de gastar
GPU. Si algo no cuadra, se detiene aqui.


In [ ]:
!python -m tools.evaluation.reevaluacion_v1 --adaptador "{ADAPTADOR_V1}" --dry-run


## 2. Reevaluacion

Primero corre el **control de reproduccion**: regenera 10 registros sin contexto
y los compara con la corrida historica. Si los 10 salen identicos, el resultado
es **REPRODUCCION**; si no, **DIAGNOSTICO** (algo del stack cambio y la cifra
indica pero no reproduce). La regla esta fijada antes de correr.

Si Colab se desconecta, volver a correr esta celda **reanuda** desde donde iba.


In [ ]:
!python -m tools.evaluation.reevaluacion_v1 --adaptador "{ADAPTADOR_V1}" --salida "{SALIDA}"


## 3. Lo que sale, y lo que falta

En `{SALIDA}/<corrida_id>/`: `afinado_results.jsonl` (las 103 respuestas, con la
historica al lado), `control_reproduccion.json` y `resumen.json`.

`resumen.json` trae la **abstencion literal** de los 35 B2 con el prompt correcto
frente a la historica. **`B2_funcional` NO sale de aqui**: necesita adjudicar las
35 respuestas nuevas con `tools/adjudicacion_b2.py`. Las adjudicaciones de
contexto existentes siguen valiendo, porque el contexto es el mismo.
